# Railway Semantic Segmentation — ResNet34 U-NetPixel-level semantic segmentation of railway scenes (rails, switches, signals, platforms, obstacles) using a **ResNet34 encoder + U-Net decoder**, trained on the **RailSem19** dataset.**Pipeline:** Download data → patchify into 768×768 tiles → train/val split → augment → train (Focal Loss, Adam, 20 epochs) → evaluate (Dice, Mean IoU) → visualise predictions.> **Note on `n_classes = 256`:** RailSem19 defines 20 semantic classes, but masks are stored as 8-bit grayscale (pixel values 0–255). This notebook uses 256 output channels to map directly onto raw pixel values without a label-encoding step. Only ~20 channels carry real labels; the rest stay empty. A cleaner implementation would label-encode masks to 20 classes — kept as-is here to reflect what was actually trained.

In [ ]:
# !pip install --upgrade tf-nightly

!pip install 'h5py==2.10.0' --force-reinstall


#Installation

In [ ]:
!git clone https://github.com/bnsreenu/python_for_microscopists.git

!pip install git+https://github.com/qubvel/segmentation_models

!pip install patchify

!pip install split-folders

!pip install focal-loss


#Libraries

In [ ]:
import os
import cv2
import glob
import numpy as np
from matplotlib import pyplot as plt
from patchify import patchify
import tifffile as tiff
from PIL import Image
import tensorflow as tf
from tensorflow import keras
import segmentation_models as sm
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.optimizers import Adam
from focal_loss import BinaryFocalLoss


sm.set_framework('tf.keras')
sm.framework()

from tensorflow.keras.metrics import MeanIoU
from tensorflow.keras.utils import normalize

import random


#Dataset

In [ ]:
# RailSem19 demo subsets hosted on Google Drive.
# Numbers = approximate image count in each subset (10 / 100 / 1000 / 4000 / full).
# Uncomment the one you want. Larger subset = better accuracy but longer training.

# 10 images
# !gdown https://drive.google.com/u/0/uc?id=1P0ldUvoNK4rHw7QieRupMye_7707kq8T&export=download

# 100 images
# !gdown https://drive.google.com/u/0/uc?id=1O6shv5hJlHwTdGlR3qwFY7IXQGQFE3-Z&export=download

# 1000 images
# !gdown https://drive.google.com/u/0/uc?id=1IRCG5DGYdEjAHaUKlkwWGEp8l4T14ge1&export=download

# 4000 images (used here)
!gdown https://drive.google.com/u/0/uc?id=1jVd0lo8eg0npQrTm-o8zh9gubnUZbrRg&export=download

# Full dataset (8500 images)
# !gdown https://drive.google.com/u/0/uc?id=1zEf38Pb6_RX6j4xYwWhSb9UBkEbMKiNL&export=download
# !unzip -qx /content/Railsem19-Dataset-Demo-Alle.zip


In [ ]:
!jar xf /content/Railsem19-Dataset-Demo-4000.zip

In [ ]:
print(len(os.listdir('/content/Railsem19-Dataset-Demo/images')))
print(len(os.listdir('/content/Railsem19-Dataset-Demo/masks')))

In [ ]:
#Quick understanding of the dataset
temp_img = cv2.imread("/content/Railsem19-Dataset-Demo/images/rs00009.jpg") #3 channels / spectral bands
plt.imshow(temp_img[:,:,2]) #View each channel...
temp_mask = cv2.imread("/content/Railsem19-Dataset-Demo/masks/rs00009.png") #3 channels but all same.
labels, count = np.unique(temp_mask[:,:,0], return_counts=True) #Check for each channel. All chanels are identical
print("Labels are: ", labels, " and the counts are: ", count)


#Patchify

In [ ]:
# NOTE: folders are named '256_patches' for historical reasons,
# but patches are actually 768x768 (see patch_size below).

directory1 = "256_patches"
parent_dir1 = "/content/Railsem19-Dataset-Demo/"
path1 = os.path.join(parent_dir1, directory1)
os.mkdir(path1)
print("Directory '%s' created" %directory1)
# ------------------------------------------------------------------------------
directory2 = "images"
parent_dir2 = "/content/Railsem19-Dataset-Demo/256_patches/"
path2 = os.path.join(parent_dir2, directory2)
os.mkdir(path2)
print("Directory '%s' created" %directory2)
# ------------------------------------------------------------------------------
directory3 = "masks"
parent_dir3 = "/content/Railsem19-Dataset-Demo/256_patches/"
path3 = os.path.join(parent_dir3, directory3)
os.mkdir(path3)
print("Directory '%s' created" %directory3)
# ------------------------------------------------------------------------------
# directory4 = "masks_input"
# parent_dir4 = "/content/Railsem19-Dataset-Demo/256_patches/"
# path4 = os.path.join(parent_dir4, directory4)
# os.mkdir(path4)
# print("Directory '%s' created" %directory4)

##Patchify Images

In [ ]:

#Now, crop each large image into patches of 256x256. Save them into a directory
#so we can use data augmentation and read directly from the drive.
root_directory = '/content/Railsem19-Dataset-Demo/'

patch_size = 768

#Read images from repsective 'images' subdirectory
#As all images are of different size we have 2 options, either resize or crop
#But, some images are too large and some small. Resizing will change the size of real objects.
#Therefore, we will crop them to a nearest size divisible by 256 and then
#divide all images into patches of 256x256x3.
img_dir=root_directory+"images/"
for path, subdirs, files in os.walk(img_dir):
    #print(path)
    dirname = path.split(os.path.sep)[-1]
    #print(dirname)
    images = sorted(os.listdir(path))  #List of all image names in this subdirectory
    #print(images)
    for i, image_name in enumerate(images):
        if image_name.endswith(".jpg"):
            #print(image_name)
            image = cv2.imread(path+"/"+image_name, 1)  #Read each image as BGR
            SIZE_X = (image.shape[1]//patch_size)*patch_size #Nearest size divisible by our patch size
            SIZE_Y = (image.shape[0]//patch_size)*patch_size #Nearest size divisible by our patch size
            image = Image.fromarray(image)
            image = image.crop((0 ,0, SIZE_X, SIZE_Y))  #Crop from top left corner
            #image = image.resize((SIZE_X, SIZE_Y))  #Try not to resize for semantic segmentation
            image = np.array(image)

            #Extract patches from each image
            print("Now patchifying image:", path+"/"+image_name)
            patches_img = patchify(image, (768, 768, 3), step=768)  #Step=256 for 256 patches means no overlap

            for i in range(patches_img.shape[0]):
                for j in range(patches_img.shape[1]):

                    single_patch_img = patches_img[i,j,:,:]
                    #single_patch_img = (single_patch_img.astype('float32')) / 255. #We will preprocess using one of the backbones
                    single_patch_img = single_patch_img[0] #Drop the extra unecessary dimension that patchify adds.

                    cv2.imwrite(root_directory+"256_patches/images/"+
                               image_name+"patch_"+str(i)+str(j)+".jpg", single_patch_img)
                    #image_dataset.append(single_patch_img)


##Patchify Masks

In [ ]:
mask_dir=root_directory+"masks/"
for path, subdirs, files in os.walk(mask_dir):
    #print(path)
    dirname = path.split(os.path.sep)[-1]

    masks = sorted(os.listdir(path))  #List of all image names in this subdirectory
    for i, mask_name in enumerate(masks):
        if mask_name.endswith(".png"):
            mask = cv2.imread(path+"/"+mask_name, 0)  #Read each image as Grey (or color but remember to map each color to an integer)
            SIZE_X = (mask.shape[1]//patch_size)*patch_size #Nearest size divisible by our patch size
            SIZE_Y = (mask.shape[0]//patch_size)*patch_size #Nearest size divisible by our patch size
            mask = Image.fromarray(mask)
            mask = mask.crop((0 ,0, SIZE_X, SIZE_Y))  #Crop from top left corner
            #mask = mask.resize((SIZE_X, SIZE_Y))  #Try not to resize for semantic segmentation
            mask = np.array(mask)

            #Extract patches from each image
            print("Now patchifying mask:", path+"/"+mask_name)
            patches_mask = patchify(mask, (768, 768), step=768)  #Step=256 for 256 patches means no overlap

            for i in range(patches_mask.shape[0]):
                for j in range(patches_mask.shape[1]):

                    single_patch_mask = patches_mask[i,j,:,:]
                    #single_patch_img = (single_patch_img.astype('float32')) / 255. #No need to scale masks, but you can do it if you want
                    #single_patch_mask = single_patch_mask[0] #Drop the extra unecessary dimension that patchify adds.
                    cv2.imwrite(root_directory+"256_patches/masks/"+
                               mask_name+"patch_"+str(i)+str(j)+".png", single_patch_mask)


In [ ]:
sorted(os.listdir('/content/Railsem19-Dataset-Demo/256_patches/images'))

In [ ]:
sorted(os.listdir('/content/Railsem19-Dataset-Demo/256_patches/masks'))

#Sanity Check

In [ ]:
print(len(os.listdir('/content/Railsem19-Dataset-Demo/256_patches/images')))
print(len(os.listdir('/content/Railsem19-Dataset-Demo/256_patches/masks')))

In [ ]:

train_img_dir = "/content/Railsem19-Dataset-Demo/256_patches/images/"
train_mask_dir = "/content/Railsem19-Dataset-Demo/256_patches/masks/"

img_list = sorted(os.listdir(train_img_dir))
msk_list = sorted(os.listdir(train_mask_dir))

num_images = len(os.listdir(train_img_dir))


img_num = random.randint(0, num_images-1)

img_for_plot = cv2.imread(train_img_dir+img_list[img_num], 1)
img_for_plot = cv2.cvtColor(img_for_plot, cv2.COLOR_BGR2RGB)

mask_for_plot =cv2.imread(train_mask_dir+msk_list[img_num], 0)

plt.figure(figsize=(12, 8))
plt.subplot(121)
plt.imshow(img_for_plot)
plt.title('Image')
plt.subplot(122)
plt.imshow(mask_for_plot, cmap='gray')
plt.title('Mask')
plt.show()


#Split Folder

In [ ]:
import splitfolders  # or import split_folders

input_folder = '/content/Railsem19-Dataset-Demo/256_patches/'

# Split with a ratio.
# To only split into training and validation set, set a tuple to `ratio`, i.e, `(.8, .2)`.
splitfolders.ratio(input_folder, output="data2", seed=1337, ratio=(.8, .2), group_prefix=None) # default values

# Split val/test with a fixed number of items e.g. 100 for each set.
# To only split into training and validation set, use a single number to `fixed`, i.e., `10`.
# splitfolders.fixed(input_folder, output="data3", seed=1337, fixed=(100, 100), oversample=False, group_prefix=None) # default values


"""
For semantic segmentation the folder structure needs to look like below
if you want to use ImageDatagenerator.

Data/
    train_images/
                train/
                    img1, img2, img3, ......

    train_masks/
                train/
                    msk1, msk, msk3, ......

    val_images/
                val/
                    img1, img2, img3, ......

    val_masks/
                val/
                    msk1, msk, msk3, ......

    test_images/
                test/
                    img1, img2, img3, ......

    test_masks/
                test/
                    msk1, msk, msk3, ......


"""

#Passende Dateinstruktur

In [ ]:
directory4 = "data_for_keras_aug"
parent_dir4 = "/content/Railsem19-Dataset-Demo/"
path4 = os.path.join(parent_dir4, directory4)
os.mkdir(path4)
print("Directory '%s' created" %directory4)

# ------------------------------------------------------------------------------
directory5 = "train_images"
parent_dir5 = "/content/Railsem19-Dataset-Demo/data_for_keras_aug/"
path5 = os.path.join(parent_dir5, directory5)
os.mkdir(path5)
print("Directory '%s' created" %directory5)


# -----------------------------------------------------------------------------
directory6 = "train_masks"
parent_dir6 = "/content/Railsem19-Dataset-Demo/data_for_keras_aug/"
path6 = os.path.join(parent_dir6, directory6)
os.mkdir(path6)
print("Directory '%s' created" %directory6)

# ------------------------------------------------------------------------------
directory7 = "val_images"
parent_dir7 = "/content/Railsem19-Dataset-Demo/data_for_keras_aug/"
path7 = os.path.join(parent_dir7, directory7)
os.mkdir(path7)
print("Directory '%s' created" %directory7)

# -----------------------------------------------------------------------------
directory8 = "val_masks"
parent_dir8 = "/content/Railsem19-Dataset-Demo/data_for_keras_aug/"
path8 = os.path.join(parent_dir8, directory8)
os.mkdir(path8)
print("Directory '%s' created" %directory8)

# -----------------------------------------------------------------------------
directory9 = "test_images"
parent_dir9 = "/content/Railsem19-Dataset-Demo/data_for_keras_aug/"
path9 = os.path.join(parent_dir9, directory9)
os.mkdir(path9)
print("Directory '%s' created" %directory9)

# -----------------------------------------------------------------------------
directory10 = "test_masks"
parent_dir10 = "/content/Railsem19-Dataset-Demo/data_for_keras_aug/"
path10 = os.path.join(parent_dir10, directory10)
os.mkdir(path10)
print("Directory '%s' created" %directory10)

# -----------------------------------------------------------------------------
os.mkdir(os.path.join("/content/Railsem19-Dataset-Demo/data_for_keras_aug/train_images/", "train"))

# -----------------------------------------------------------------------------
os.mkdir(os.path.join("/content/Railsem19-Dataset-Demo/data_for_keras_aug/train_masks/", "train"))

# -----------------------------------------------------------------------------
os.mkdir(os.path.join("/content/Railsem19-Dataset-Demo/data_for_keras_aug/val_images/", "val"))

# -----------------------------------------------------------------------------
os.mkdir(os.path.join("/content/Railsem19-Dataset-Demo/data_for_keras_aug/val_masks/", "val"))

# -----------------------------------------------------------------------------
os.mkdir(os.path.join("/content/Railsem19-Dataset-Demo/data_for_keras_aug/test_images/", "test"))

# -----------------------------------------------------------------------------
os.mkdir(os.path.join("/content/Railsem19-Dataset-Demo/data_for_keras_aug/test_masks/", "test"))



In [ ]:
import shutil

for image in sorted(os.listdir("/content/data2/train/images")):
    shutil.move("/content/data2/train/images/"+image, "/content/Railsem19-Dataset-Demo/data_for_keras_aug/train_images/train/")

for mask in sorted(os.listdir("/content/data2/train/masks")):
    shutil.move("/content/data2/train/masks/"+mask, "/content/Railsem19-Dataset-Demo/data_for_keras_aug/train_masks/train/")

for image in sorted(os.listdir("/content/data2/val/images")):
    shutil.move("/content/data2/val/images/"+image, "/content/Railsem19-Dataset-Demo/data_for_keras_aug/val_images/val/")

for mask in sorted(os.listdir("/content/data2/val/masks")):
    shutil.move("/content/data2/val/masks/"+mask, "/content/Railsem19-Dataset-Demo/data_for_keras_aug/val_masks/val/")

# for image in sorted(os.listdir("/content/data2/test/images")):
#     shutil.move("/content/data2/test/images/"+image, "/content/Railsem19-Dataset-Demo/data_for_keras_aug/test_images/test/")

# for mask in sorted(os.listdir("/content/data2/test/masks")):
#     shutil.move("/content/data2/test/masks/"+mask, "/content/Railsem19-Dataset-Demo/data_for_keras_aug/test_masks/test/")

# Another way: (but it didn't work)

# !mv "/content/data2/train/images/*" "/content/Railsem19-Dataset-Demo/data_for_keras_aug/train_images/train/"

# !mv "/content/data2/train/masks/*" "/content/Railsem19-Dataset-Demo/data_for_keras_aug/train_masks/train"

# !mv "/content/data2/val/images/*" "/content/Railsem19-Dataset-Demo/data_for_keras_aug/val_images/val/"

# !mv "/content/data2/val/masks/*" "/content/Railsem19-Dataset-Demo/data_for_keras_aug/val_masks/val/"

# !mv "/content/data2/test/images/*" "/content/Railsem19-Dataset-Demo/data_for_keras_aug/test_images/test/"

# !mv "/content/data2/test/masks/*" "/content/Railsem19-Dataset-Demo/data_for_keras_aug/test_masks/test/"


#ImageDataGenerator

In [ ]:
train_img_dir = "/content/Railsem19-Dataset-Demo/data_for_keras_aug/train_images/train/"
train_mask_dir = "/content/Railsem19-Dataset-Demo/data_for_keras_aug/train_masks/train/"

img_list = sorted(os.listdir(train_img_dir))
msk_list = sorted(os.listdir(train_mask_dir))

num_images = len(os.listdir(train_img_dir))


img_num = random.randint(0, num_images-1)

img_for_plot = cv2.imread(train_img_dir+img_list[img_num], 1)
img_for_plot = cv2.cvtColor(img_for_plot, cv2.COLOR_BGR2RGB)

mask_for_plot =cv2.imread(train_mask_dir+msk_list[img_num], 0)

plt.figure(figsize=(12, 8))
plt.subplot(121)
plt.imshow(img_for_plot)
plt.title('Image')
plt.subplot(122)
plt.imshow(mask_for_plot, cmap='gray')
plt.title('Mask')
plt.show()

In [ ]:

# Define Generator for images and masks so we can read them directly from the drive.

seed=24
batch_size= 10
n_classes= 256

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
from tensorflow.keras.utils import to_categorical

#Use this to preprocess input for transfer learning
BACKBONE = 'resnet34'  # pretrained on ImageNet (transfer learning)
preprocess_input = sm.get_preprocessing(BACKBONE)

#Define a function to perform additional preprocessing after datagen.
#For example, scale images, convert masks to categorical, etc.
def preprocess_data(img, mask, num_class):
    #Scale images
    img = scaler.fit_transform(img.reshape(-1, img.shape[-1])).reshape(img.shape)
    # img = np.expand_dims(normalize(np.array(img), axis=1),3)
    img = preprocess_input(img)  #Preprocess based on the pretrained backbone...
    # img = to_categorical(img, num_class)
    #Convert mask to one-hot
    mask = to_categorical(mask, num_class)

    # from sklearn.preprocessing import LabelEncoder
    # labelencoder = LabelEncoder()
    # n, h, w = mask.shape
    # train_masks_reshaped = mask_dataset_uint8.reshape(-1,1)
    # train_masks_reshaped_encoded = labelencoder.fit_transform(train_masks_reshaped)
    # train_masks_encoded_original_shape = train_masks_reshaped_encoded.reshape(n, h, w)
    # train_masks_input = np.expand_dims(train_masks_encoded_original_shape, axis=3)

    return (img,mask)

#Define the generator.
#We are not doing any rotation or zoom to make sure mask values are not interpolated.
#It is important to keep pixel values in mask as 0, 1, 2, 3, .....

from tensorflow.keras.preprocessing.image import ImageDataGenerator
def trainGenerator(train_img_path, train_mask_path, num_class):

    img_data_gen_args = dict(horizontal_flip=True,
                      vertical_flip=True,
                      fill_mode='reflect')

    image_datagen = ImageDataGenerator(**img_data_gen_args)
    mask_datagen = ImageDataGenerator(**img_data_gen_args)

    image_generator = image_datagen.flow_from_directory(
        train_img_path,
        class_mode = None,
        batch_size = batch_size,
        seed = seed)

    mask_generator = mask_datagen.flow_from_directory(
        train_mask_path,
        class_mode = None,
        color_mode = 'grayscale',
        batch_size = batch_size,
        seed = seed)

    train_generator = zip(image_generator, mask_generator)

    for (img, mask) in train_generator:
        img, mask = preprocess_data(img, mask, num_class)
        yield (img, mask)



In [ ]:
train_img_path = "/content/Railsem19-Dataset-Demo/data_for_keras_aug/train_images/"
train_mask_path = "/content/Railsem19-Dataset-Demo/data_for_keras_aug/train_masks/"

train_img_gen = trainGenerator(train_img_path, train_mask_path, num_class=256)

val_img_path = "/content/Railsem19-Dataset-Demo/data_for_keras_aug/val_images/"
val_mask_path = "/content/Railsem19-Dataset-Demo/data_for_keras_aug/val_masks/"

val_img_gen = trainGenerator(val_img_path, val_mask_path, num_class=256)

#Make sure the generator is working and that images and masks are indeed lined up.
#Verify generator.... In python 3 next() is renamed as __next__()
x, y = train_img_gen.__next__()

for i in range(0,9):
    image = x[i]
    mask = np.argmax(y[i], axis=2)
    plt.subplot(1,2,1)
    plt.imshow(image)
    plt.subplot(1,2,2)
    plt.imshow(mask, cmap='gray')
    plt.show()

x_val, y_val = val_img_gen.__next__()

for i in range(0,9):
    image = x_val[i]
    mask = np.argmax(y_val[i], axis=2)
    plt.subplot(1,2,1)
    plt.imshow(image)
    plt.subplot(1,2,2)
    plt.imshow(mask, cmap='gray')
    plt.show()


In [ ]:
x.shape

In [ ]:
y.shape

#Definde the Model

In [ ]:
#Define the model metrcis and load model.

num_train_imgs = len(os.listdir('/content/Railsem19-Dataset-Demo/data_for_keras_aug/train_images/train/'))
num_val_images = len(os.listdir('/content/Railsem19-Dataset-Demo/data_for_keras_aug/val_images/val/'))
steps_per_epoch = num_train_imgs//batch_size
val_steps_per_epoch = num_val_images//batch_size


IMG_HEIGHT = x.shape[1]
IMG_WIDTH  = x.shape[2]
IMG_CHANNELS = x.shape[3]

n_classes= 256


#Model

In [ ]:
#Jaccard distance loss mimics IoU.
from keras import backend as K
def jaccard_distance_loss(y_true, y_pred, smooth=100):
    """
    Jaccard = (|X & Y|)/ (|X|+ |Y| - |X & Y|)
            = sum(|A*B|)/(sum(|A|)+sum(|B|)-sum(|A*B|))

    The jaccard distance loss is usefull for unbalanced datasets. This has been
    shifted so it converges on 0 and is smoothed to avoid exploding or disapearing
    gradient.

    Ref: https://en.wikipedia.org/wiki/Jaccard_index

    @url: https://gist.github.com/wassname/f1452b748efcbeb4cb9b1d059dce6f96
    @author: wassname
    """
    intersection = K.sum(K.sum(K.abs(y_true * y_pred), axis=-1))
    sum_ = K.sum(K.sum(K.abs(y_true) + K.abs(y_pred), axis=-1))
    jac = (intersection + smooth) / (sum_ - intersection + smooth)
    return (1 - jac) * smooth

#Dice metric can be a great metric to track accuracy of semantic segmentation.
def dice_metric(y_pred, y_true):
    intersection = K.sum(K.sum(K.abs(y_true * y_pred), axis=-1))
    union = K.sum(K.sum(K.abs(y_true) + K.abs(y_pred), axis=-1))
    # if y_pred.sum() == 0 and y_pred.sum() == 0:
    #     return 1.0

    return 2*intersection / union


In [ ]:
#Define the model
# define model

model = sm.Unet(BACKBONE, encoder_weights='imagenet',
                input_shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS),
                classes=n_classes, activation='softmax')
model.compile(optimizer=Adam(lr = 1e-3), loss=BinaryFocalLoss(gamma=2), metrics=[dice_metric])





#Other losses to try: categorical_focal_dice_loss, cce_jaccard_loss, cce_dice_loss, categorical_focal_loss

#model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=metrics)
print(model.summary())
print(model.input_shape)



#Train

In [ ]:
# Train the model
history = model.fit(train_img_gen,
                    steps_per_epoch=steps_per_epoch,
                    epochs=20,
                    verbose=1,
                    validation_data=val_img_gen,
                    validation_steps=val_steps_per_epoch)

model.save('railsem19_resnet34_20epochs_768patch.hdf5')


#Evaluation

In [ ]:
#plot the training and validation IoU and loss at each epoch
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'y', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

acc = history.history['dice_metric']
val_acc = history.history['val_dice_metric']

plt.plot(epochs, acc, 'y', label='Training Dice')
plt.plot(epochs, val_acc, 'r', label='Validation Dice')
plt.title('Training and validation IoU')
plt.xlabel('Epochs')
plt.ylabel('Dice')
plt.legend()
plt.show()


In [ ]:
from keras.models import load_model

model = load_model("railsem19_resnet34_20epochs_768patch.hdf5", compile=False)

#batch_size=32 #Check IoU for a batch of images

#Test generator using validation data.

test_image_batch, test_mask_batch = val_img_gen.__next__()

#Convert categorical to integer for visualization and IoU calculation
test_mask_batch_argmax = np.argmax(test_mask_batch, axis=3)
test_pred_batch = model.predict(test_image_batch)
test_pred_batch_argmax = np.argmax(test_pred_batch, axis=3)

n_classes = 256
IOU_keras = MeanIoU(num_classes=n_classes)
IOU_keras.update_state(test_pred_batch_argmax, test_mask_batch_argmax)
print("Mean IoU =", IOU_keras.result().numpy())


In [ ]:
#View a few images, masks and corresponding predictions.
img_num = random.randint(0, test_image_batch.shape[0]-1)

plt.figure(figsize=(12, 8))
plt.subplot(231)
plt.title('Testing Image')
plt.imshow(test_image_batch[img_num])
plt.subplot(232)
plt.title('Testing Label')
plt.imshow(test_mask_batch_argmax[img_num])
plt.subplot(233)
plt.title('Prediction on test image')
plt.imshow(test_pred_batch_argmax[img_num])
plt.show()
